# SIFT Features + Gradient Boosting — Image Classification Pipeline

**Feature descriptor:** SIFT + Bag of Visual Words (100 words) &nbsp;|&nbsp; **Classifier:** Gradient Boosting Classifier &nbsp;|&nbsp; **Validation:** 5-fold cross-validation

---

This notebook implements an end-to-end image-classification experiment:

1. **Feature Extraction** — SIFT keypoint descriptors are extracted per image, then encoded as fixed-length Bag-of-Visual-Words histograms using a 100-word KMeans vocabulary learned from the training folds.
2. **Bag of Visual Words** — per-image descriptor sets are quantized against a KMeans vocabulary into fixed-length, normalized word histograms.
3. **Data Loading** — images are read fold-by-fold from a directory structure of `train` / `test` splits.
4. **Model Training & Evaluation** — a **Gradient Boosting classifier** — `GradientBoostingClassifier(random_state=42)` is trained on each fold, across six image resolutions, and evaluated with accuracy, precision, recall, and F1 (weighted & macro).
5. **Results** — per-fold metrics, 5-fold averages, and confusion matrices are reported; results are exported to CSV.

**Outputs**

| File | Contents |
|---|---|
| `SIFT_GradientBoosting_5Fold_All_Results.csv` | Metrics for every fold x image size |
| `SIFT_GradientBoosting_5Fold_Average_Results.csv` | Metrics averaged over the 5 folds |


## 1&nbsp;&nbsp;Imports & Logging

Standard scientific-Python stack: OpenCV and scikit-image for image processing, scikit-learn for the classifier and metrics, and matplotlib/seaborn for visualization. Logging is configured to report progress and any per-image failures without halting the experiment.


In [ ]:
import os
import logging
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from joblib import Parallel, delayed

from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)


# ======================================================
# LOGGING
# ======================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)




## 2&nbsp;&nbsp;SIFT Feature Extraction

`extract_sift_descriptors(image, target_size)` converts the image to grayscale, resizes it to `target_size`, and runs OpenCV's `SIFT_create().detectAndCompute()` to detect keypoints and return their **128-dimensional SIFT descriptors** (a variable number per image). Extraction failures are logged and the image is skipped.


In [ ]:
# ======================================================
# SIFT FEATURE EXTRACTION
# ======================================================

def extract_sift_descriptors(image, target_size):
    try:
        # Convert to grayscale
        if image.ndim == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image

        # Resize image
        gray = cv2.resize(gray, target_size)

        # Create SIFT detector
        sift = cv2.SIFT_create()

        # Detect keypoints and descriptors
        keypoints, descriptors = sift.detectAndCompute(gray, None)

        return descriptors

    except Exception as e:
        logging.error(f"SIFT extraction failed: {e}")
        return None




## 3&nbsp;&nbsp;Descriptor Loading

`load_descriptors(directory, target_size)` discovers class sub-directories, reads each image with OpenCV (converting BGR to RGB), extracts its SIFT descriptors, and returns the per-image descriptor list, integer-encoded labels, and the ordered class names. Images with no detectable keypoints are skipped.


In [ ]:
# ======================================================
# LOAD IMAGES AND EXTRACT DESCRIPTORS
# ======================================================

def load_descriptors(directory, target_size):

    class_names = sorted([
        d for d in os.listdir(directory)
        if os.path.isdir(os.path.join(directory, d))
    ])

    image_descriptors = []
    labels = []

    for class_name in class_names:

        class_path = os.path.join(directory, class_name)

        for file_name in tqdm(
            os.listdir(class_path),
            desc=f"Loading {class_name}"
        ):

            if file_name.startswith("."):
                continue

            image_path = os.path.join(class_path, file_name)

            image = cv2.imread(image_path)

            if image is None:
                continue

            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            descriptors = extract_sift_descriptors(image, target_size)

            if descriptors is not None and len(descriptors) > 0:
                image_descriptors.append(descriptors)
                labels.append(class_names.index(class_name))

    return image_descriptors, np.array(labels), class_names




## 4&nbsp;&nbsp;Bag of Visual Words Encoding

`build_bovw_features(descriptor_list, kmeans)` encodes each image as a fixed-length **Bag of Visual Words** histogram: every descriptor is assigned to its nearest KMeans cluster center (visual word), word counts are accumulated, and the histogram is L1-normalized. This turns variable-length descriptor sets into uniform feature vectors suitable for classification.


In [ ]:
# ======================================================
# BUILD BAG OF VISUAL WORDS FEATURES
# ======================================================

def build_bovw_features(descriptor_list, kmeans):

    n_clusters = kmeans.n_clusters
    features = np.zeros((len(descriptor_list), n_clusters), dtype=np.float32)

    for i, descriptors in enumerate(descriptor_list):
        if descriptors is not None and len(descriptors) > 0:

            words = kmeans.predict(descriptors)

            for w in words:
                features[i, w] += 1

            # Normalize histogram
            features[i] /= np.sum(features[i])

    return features




## 5&nbsp;&nbsp;Experiment Configuration

Defines the dataset location, the five cross-validation folds, the six image resolutions to be evaluated, and the visual-vocabulary size (`NUM_VISUAL_WORDS = 100`). `all_results` accumulates the metrics from every (image size, fold) run.


In [ ]:
# ======================================================
# 5-FOLD SETTINGS
# ======================================================

baseDir = r"E:\THUSHAR\DATASET\Croped_5Fold"

# For Google Colab use:
# baseDir = "/content/drive/MyDrive/Croped_5Fold"

folds = [
    "fold_1",
    "fold_2",
    "fold_3",
    "fold_4",
    "fold_5"
]


# ======================================================
# IMAGE SIZES
# ======================================================

sizes = [
    (8, 8),
    (16, 16),
    (32, 32),
    (64, 64),
    (128, 128),
    (196, 210)
]


# ======================================================
# NUMBER OF VISUAL WORDS
# ======================================================

NUM_VISUAL_WORDS = 100


# ======================================================
# STORE RESULTS
# ======================================================

all_results = []




## 6&nbsp;&nbsp;Training & Evaluation — Main Experiment Loop

For every image size and every fold:

1. Load train and test SIFT descriptors for the fold.
2. Build the **visual vocabulary**: fit KMeans (100 clusters) on all stacked training descriptors.
3. Encode train and test images as normalized Bag-of-Visual-Words histograms.
4. Train a **Gradient Boosting classifier** — `GradientBoostingClassifier(random_state=42)`.
5. Predict on the test set and compute **accuracy**, **precision / recall / F1** (weighted and macro).
6. Append the metrics to `all_results` and plot the **confusion matrix**.


In [ ]:
# ======================================================
# MAIN EXPERIMENT LOOP
# ======================================================

for size in sizes:

    print("\n========================================")
    print(f"PROCESSING IMAGE SIZE: {size[0]}x{size[1]}")
    print("========================================")

    for fold in folds:

        print(f"\n########## {fold} ##########")

        mainDir = os.path.join(baseDir, fold)

        # --------------------------------------------------
        # LOAD TRAIN DESCRIPTORS
        # --------------------------------------------------

        train_desc, y_train, class_names = load_descriptors(
            os.path.join(mainDir, "train"),
            size
        )

        test_desc, y_test, _ = load_descriptors(
            os.path.join(mainDir, "test"),
            size
        )

        # --------------------------------------------------
        # CREATE VISUAL VOCABULARY
        # --------------------------------------------------

        all_train_descriptors = np.vstack(train_desc)

        print("Building visual vocabulary...")

        kmeans = MiniBatchKMeans(
            n_clusters=NUM_VISUAL_WORDS,
            random_state=42,
            batch_size=1000
        )

        kmeans.fit(all_train_descriptors)

        # --------------------------------------------------
        # CONVERT TO FIXED-LENGTH FEATURES
        # --------------------------------------------------

        X_train = build_bovw_features(train_desc, kmeans)
        X_test = build_bovw_features(test_desc, kmeans)

        print("Train shape:", X_train.shape)
        print("Test shape :", X_test.shape)

        # --------------------------------------------------
        # GRADIENT BOOSTING CLASSIFIER
        # --------------------------------------------------

        clf = GradientBoostingClassifier(
            random_state=42
        )

        # --------------------------------------------------
        # TRAIN
        # --------------------------------------------------

        print("\nTraining Gradient Boosting ...")

        clf.fit(X_train, y_train)

        # --------------------------------------------------
        # PREDICT
        # --------------------------------------------------

        y_pred = clf.predict(X_test)

        # ==================================================
        # METRICS
        # ==================================================

        accuracy = accuracy_score(y_test, y_pred)

        precision_w, recall_w, f1_w, _ = (
            precision_recall_fscore_support(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_m, recall_m, f1_m, _ = (
            precision_recall_fscore_support(
                y_test,
                y_pred,
                average="macro",
                zero_division=0
            )
        )

        print(f"Accuracy            : {accuracy:.4f}")
        print(f"F1 Weighted         : {f1_w:.4f}")
        print(f"F1 Macro            : {f1_m:.4f}")

        # ==================================================
        # SAVE RESULTS
        # ==================================================

        all_results.append({
            "Fold": fold,
            "Feature": "SIFT",
            "Classifier": "GradientBoosting",
            "Image_Size": f"{size[0]}x{size[1]}",
            "Accuracy": accuracy,
            "Precision_Weighted": precision_w,
            "Recall_Weighted": recall_w,
            "F1_Weighted": f1_w,
            "Precision_Macro": precision_m,
            "Recall_Macro": recall_m,
            "F1_Macro": f1_m
        })

        # --------------------------------------------------
        # CONFUSION MATRIX
        # --------------------------------------------------

        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(10, 8))

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names
        )

        plt.title(
            f"SIFT + Gradient Boosting\n{fold} ({size[0]}x{size[1]})"
        )

        plt.xlabel("Predicted Class")
        plt.ylabel("Actual Class")
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()




## 7&nbsp;&nbsp;Results Aggregation & Export

Fold-wise metrics are saved to CSV, averaged across the five folds per image size, exported, and displayed sorted by image size and accuracy.


In [ ]:
# ======================================================
# SAVE FOLD-WISE RESULTS
# ======================================================

results_df = pd.DataFrame(all_results)

results_df.to_csv(
    "SIFT_GradientBoosting_5Fold_All_Results.csv",
    index=False
)

print(
    "\nFold-wise results saved: SIFT_GradientBoosting_5Fold_All_Results.csv"
)


# ======================================================
# COMPUTE AVERAGE RESULTS
# ======================================================

average_df = results_df.groupby(
    ["Feature", "Classifier", "Image_Size"]
).agg({
    "Accuracy": "mean",
    "Precision_Weighted": "mean",
    "Recall_Weighted": "mean",
    "F1_Weighted": "mean",
    "Precision_Macro": "mean",
    "Recall_Macro": "mean",
    "F1_Macro": "mean"
}).reset_index()

average_df.to_csv(
    "SIFT_GradientBoosting_5Fold_Average_Results.csv",
    index=False
)

print(
    "Average results saved: SIFT_GradientBoosting_5Fold_Average_Results.csv"
)


# ======================================================
# DISPLAY FINAL RESULTS
# ======================================================

print("\n========================================")
print("FINAL 5-FOLD AVERAGE RESULTS")
print("========================================")

print(
    average_df.sort_values(
        ["Image_Size", "Accuracy"],
        ascending=[True, False]
    )
)

print("\n========================================")
print("SIFT + GRADIENT BOOSTING EXPERIMENTS COMPLETED SUCCESSFULLY")
print("========================================")